# Fundable - Company Funding & Lead Lookup

**Option 1:** Search by filters (funding recency, company size, industry, person location)  
**Option 2:** Search by company URLs/LinkedIn

**Output:**  
- `company_df` — one row per company (funding details, investors, HQ)  
- `leads_df` — all available leads per company (contact + company + funding)

In [ ]:
import requests
import json
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta
import os

load_dotenv()

API_KEY = os.environ.get("TRYFUNDABLE_API_KEY", "")
BASE_URL = "https://www.tryfundable.ai/api/v1"
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Accept": "application/json"}

# Load mapping file
with open("fundable_mapping.json") as f:
    FUNDABLE_MAP = json.load(f)

# Location normalization using geo_lookup_fast.json
from geo_normalize import normalize_location as _geo_normalize, infer_country_from_state, normalize_country

# Load area-city mappings (186 metro areas with state/country)
with open('area_city_mappings.json', encoding='utf-8') as _af:
    _AREA_MAPPINGS = json.load(_af).get('mappings', {})
# Build normalized lookup: lowercase area name -> (state, country)
# Build normalized lookup using normalize_area_name
def _norm_area(a):
    a = a.lower().strip()
    a = a.replace("greater ", "").replace(" metropolitan", "").replace(" metro", "").replace(" area", "")
    import re
    a = re.sub(r"\s*(metroplex|bay area|region)$", "", a).strip()
    return a
_AREA_LOOKUP = {_norm_area(k): v for k, v in _AREA_MAPPINGS.items()}
# Also keep original keys for city list lookup
_AREA_NORM_TO_KEY = {_norm_area(k): k for k in _AREA_MAPPINGS}
# Build reverse: city -> (state, country) from area mappings
_CITY_TO_AREA = {}
for _area_info in _AREA_MAPPINGS.values():
    for _c in _area_info.get('cities', []):
        _CITY_TO_AREA[_c.lower()] = (_area_info.get('state', ''), _area_info.get('country', ''))
del _af

# Load intl cities for fallback
with open('geo_lookup_fast.json', encoding='utf-8') as _gf:
    _geo_raw = json.load(_gf)
_INTL_CITIES = {co: set(cities) for co, cities in _geo_raw.get('cities', {}).items()}
del _gf, _geo_raw

def normalize_area_name(area):
    """Normalize area name for matching. e.g. "Greater Seattle Area" -> "seattle"."""
    import re as _re
    area = area.lower().strip()
    area = area.replace("greater ", "").replace(" metropolitan", "").replace(" metro", "").replace(" area", "")
    area = _re.sub(r"\s*(metroplex|bay area|region)$", "", area).strip()
    return area

def normalize_location(city_raw):
    """Parse person city field into (city, state, country). Uses area_city_mappings + geo_lookup_fast."""
    if not city_raw:
        return "", "", ""
    city_raw = city_raw.strip()
    import re as _re
    # 1. Check exact area name match (e.g. "San Francisco Bay Area")
    area_norm = _norm_area(city_raw)
    if area_norm in _AREA_LOOKUP:
        info = _AREA_LOOKUP[area_norm]
        # Extract main city from area name (strip Greater/Metro/Area suffixes)
        main_city = _re.sub(r'^(Metro|Greater)\s+', '', city_raw, flags=_re.I).strip()
        main_city = _re.sub(r'\s*(Metropolitan Area|Metropolitan|Metroplex|Metro Area|Metro|Bay Area|Area|Region)$', '', main_city, flags=_re.I).strip()
        # For hyphenated names, take first city
        if '-' in main_city:
            main_city = main_city.split('-')[0].strip()
        return main_city, info.get('state', ''), info.get('country', '')
    # 2. Strip metro/area prefixes and suffixes
    cleaned = _re.sub(r"^(Metro|Greater)\s+", "", city_raw, flags=_re.I).strip()
    cleaned = _re.sub(r"\s*(Metropolitan Area|Metropolitan|Metroplex|Metro Area|Metro|Bay Area|Area|Region)$", "", cleaned, flags=_re.I).strip()
    # 3. For hyphenated metros, take first city
    if "-" in cleaned and "," not in cleaned:
        first_city = cleaned.split("-")[0].strip()
        if first_city.lower() in _CITY_TO_AREA:
            cleaned = first_city
    # 4. Split by comma
    parts = [x.strip() for x in cleaned.split(",")]
    raw_city = parts[0] if len(parts) >= 1 else ""
    raw_state = parts[1] if len(parts) >= 2 else ""
    raw_country = parts[2] if len(parts) >= 3 else ""
    # 5. Single city (no comma) — check area city reverse lookup
    if not raw_state and not raw_country:
        city_lower = raw_city.lower()
        if city_lower in _CITY_TO_AREA:
            state, country = _CITY_TO_AREA[city_lower]
            return raw_city.title(), state, country
        # Check international cities
        for co, city_set in _INTL_CITIES.items():
            if city_lower in city_set:
                return raw_city.title(), "", co.title()
    # 6. Standard format — use geo_normalize
    city, state, country = _geo_normalize(raw_city, raw_state, raw_country)
    if country:
        country = normalize_country(country)
    if not country and state:
        country = infer_country_from_state(state)
    return city, state, country

def to_industry_slugs(names):
    """Convert taxonomy names like 'SaaS,FinTech' to Fundable slugs."""
    if not names:
        return ""
    slugs = []
    for name in names.split(","):
        name = name.strip()
        slug = FUNDABLE_MAP["industries"].get(name)
        if slug:
            slugs.append(slug)
        else:
            print(f"  Warning: '{name}' not found in mapping, skipping")
    return ",".join(slugs)

def to_employee_slugs(sizes):
    """Convert taxonomy sizes like '51-200' to Fundable slugs."""
    if not sizes:
        return ""
    slugs = []
    for size in sizes.split(","):
        size = size.strip()
        mapped = FUNDABLE_MAP["employee_counts"].get(size)
        if mapped:
            for s in mapped.split(","):
                if s not in slugs:
                    slugs.append(s)
        else:
            print(f"  Warning: '{size}' not found in mapping, skipping")
    return ",".join(slugs)

print(f"API Key configured: {bool(API_KEY)}")
print(f"Mapping loaded: {len(FUNDABLE_MAP['industries'])} industries, {len(FUNDABLE_MAP['employee_counts'])} employee counts")

In [ ]:
import re

SCRAPINGDOG_API_KEY = os.environ.get("SCRAPINGDOG_API_KEY", "")

# LinkedIn standard ranges
LINKEDIN_RANGES = [
    ((0, 1), "0-1"), ((2, 10), "2-10"), ((11, 50), "11-50"),
    ((51, 200), "51-200"), ((201, 500), "201-500"), ((501, 1000), "501-1,000"),
    ((1001, 5000), "1,001-5,000"), ((5001, 10000), "5,001-10,000"),
    ((10001, 100000), "10,001+"),
]

def parse_employee_count(text):
    if not text:
        return None
    text = text.strip().replace(",", "")
    if text.endswith("+"):
        val = int(re.sub(r'[^0-9]', '', text))
        return (val, 100000)
    parts = re.split(r'[-–—]', text)
    if len(parts) == 2:
        try:
            return (int(parts[0].strip()), int(parts[1].strip()))
        except ValueError:
            return None
    try:
        return (int(text), int(text))
    except ValueError:
        return None

def normalize_to_linkedin_range(min_val, max_val):
    for (r_min, r_max), r_str in LINKEDIN_RANGES:
        if min_val == r_min and (max_val == r_max or (r_str == "10,001+" and max_val >= 10001)):
            return r_str
        if min_val == max_val and r_min <= min_val <= r_max:
            return r_str
    return None

def does_size_match_user_input(linkedin_range_str, user_size_input):
    if not user_size_input or not linkedin_range_str:
        return True
    ext_parsed = parse_employee_count(linkedin_range_str)
    if not ext_parsed:
        return True
    ext_min, ext_max = ext_parsed
    for user_range in user_size_input.split(","):
        user_range = user_range.strip()
        user_parsed = parse_employee_count(user_range)
        if not user_parsed:
            continue
        u_min, u_max = user_parsed
        if ext_min >= u_min and ext_max <= u_max:
            return True
        if ext_min == u_min and ext_max == u_max:
            return True
        if u_min <= ext_min <= u_max or u_min <= ext_max <= u_max:
            return True
    return False

def extract_employee_count_from_results(search_results, company="", company_slug=""):
    patterns = [
        r'company\s*size[:\s]+(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*employees?',
        r'company\s*size[:\s]+(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*|\d+\+)',
        r'(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*employees?',
        r'(\d+[\s,-–—]+\d+)\s*employees?',
        r'(\d{1,3}(?:,\d{3})*\+)\s*employees?',
        r'(\d+\+|\d+[\s,-–—]+\d+)\s*employees?',
        r'·\s*(\d{1,2}[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*employees?',
        r'(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*mitarbeiter',
        r'(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*employés',
        r'(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*empleados',
        r'(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*dipendenti',
        r'(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*(?:werknemers|medewerkers)',
        r'(\d{1,3}(?:,\d{3})*[\s,-–—]+\d{1,3}(?:,\d{3})*)\s*funcionários',
        r'·\s*(2[-–]10|11[-–]50|51[-–]200|201[-–]500|501[-–]1,?000|1,?001[-–]5,?000|5,?001[-–]10,?000|10,?001\+)',
    ]
    for result in search_results:
        title = result.get("title", "")
        snippet = result.get("body", result.get("snippet", ""))
        href = result.get("href", "").lower()
        if "linkedin.com" not in href:
            continue
        if company_slug:
            slug_match = False
            for pat in [f'/company/{company_slug}/', f'/company/{company_slug}?', f'/company/{company_slug}#']:
                if pat in href:
                    slug_match = True
                    break
            if href.endswith(f'/company/{company_slug}'):
                slug_match = True
            if not slug_match:
                continue
        combined = f"{title} {snippet}".lower()
        for pattern in patterns:
            match = re.search(pattern, combined, re.IGNORECASE)
            if match:
                extracted = match.group(1).strip()
                extracted = re.sub(r'[\s]+', '', extracted)
                extracted = re.sub(r'[–—]', '-', extracted)
                clean = extracted.replace(",", "").replace("+", "")
                if "-" in clean:
                    parts = clean.split("-")
                    try:
                        if int(parts[0]) > 0 and int(parts[1]) >= int(parts[0]):
                            return extracted
                    except ValueError:
                        pass
                elif clean.isdigit() and int(clean) > 0:
                    return extracted
    return None

def verify_employee_size_via_linkedin(company_linkedin_url, company_name, user_size, fundable_employee_range=""):
    if not SCRAPINGDOG_API_KEY:
        return None, True, "no_api_key"
    slug = ""
    if company_linkedin_url:
        match = re.search(r'linkedin\.com/company/([^/?#]+)', company_linkedin_url)
        if match:
            slug = match.group(1).strip("/")
    if not slug:
        return None, True, "no_slug"

    # Same query order as Leadpoet stage 5 + linkedin.com/company/{slug} "size"
    # Build query 3: convert Fundable's range to LinkedIn/taxonomy format
    # Reverse map: Fundable slug -> taxonomy range (which matches LinkedIn format)
    FUNDABLE_TO_LINKEDIN = {
        "1-10": "2-10", "11-50": "11-50",
        "51-100": "51-200", "101-250": "51-200",
        "251-500": "201-500", "501-1000": "501-1,000",
        "1001-5000": "1,001-5,000", "5001-10000": "5,001-10,000",
        "10001+": "10,001+",
    }
    linkedin_size = FUNDABLE_TO_LINKEDIN.get(fundable_employee_range or "", "")

    queries = [
        f'linkedin.com/company/{slug}+size',
        f'linkedin.com/company/{slug}+Company size',
    ]
    if linkedin_size:
        queries.append(f'linkedin.com/company/{slug} "{linkedin_size}"')

    for query in queries:
        try:
            resp = requests.get("https://api.scrapingdog.com/google", params={
                "api_key": SCRAPINGDOG_API_KEY, "query": query, "results": 5
            }, timeout=30)
            if resp.status_code == 200:
                results = [{"title": r.get("title",""), "href": r.get("link",""), "body": r.get("snippet","")}
                           for r in resp.json().get("organic_results", [])]
                extracted = extract_employee_count_from_results(results, company_name, slug)
                if extracted:
                    ext_parsed = parse_employee_count(extracted)
                    if ext_parsed:
                        ext_range = normalize_to_linkedin_range(*ext_parsed) or extracted
                        size_match = does_size_match_user_input(ext_range, user_size)
                        return ext_range, size_match, "linkedin"
        except Exception:
            pass

    return None, True, "scrape_failed"



def extract_hq_from_linkedin(company_linkedin_url, company_name):
    """Extract headquarters from LinkedIn via ScrapingDog when Fundable HQ is empty."""
    if not SCRAPINGDOG_API_KEY:
        return "", "", ""
    slug = ""
    if company_linkedin_url:
        match = re.search(r'linkedin\.com/company/([^/?#]+)', company_linkedin_url)
        if match:
            slug = match.group(1).strip("/")
    if not slug:
        return "", "", ""

    queries = [
        f'linkedin.com/company/{slug}+Headquarters',
        f'linkedin.com/company/{slug} "Headquarters"',
    ]
    for query in queries:
        try:
            resp = requests.get("https://api.scrapingdog.com/google", params={
                "api_key": SCRAPINGDOG_API_KEY, "query": query, "results": 5
            }, timeout=30)
            if resp.status_code == 200:
                for r in resp.json().get("organic_results", []):
                    link = r.get("link", "").lower()
                    if f'/company/{slug}' not in link:
                        continue
                    snippet = r.get("snippet", "")
                    m = re.search(r'[Hh]eadquarters[:\s]+([A-Z][^.\xb7|\n]+)', snippet)
                    if m:
                        hq = m.group(1).strip().rstrip(".")
                        parts = [p.strip() for p in hq.split(",")]
                        city = parts[0] if len(parts) >= 1 else ""
                        state = parts[1] if len(parts) >= 2 else ""
                        country = parts[2] if len(parts) >= 3 else ""
                        if city and len(city) > 2:
                            return city, state, country
        except Exception:
            pass
    return "", "", ""


def resolve_company(url):
    url = url.strip()
    if "linkedin.com" in url:
        param = {"linkedin": url}
    else:
        domain = url.replace("https://", "").replace("http://", "").replace("www.", "").rstrip("/")
        param = {"domain": domain}
    resp = requests.get(f"{BASE_URL}/company/", headers=HEADERS, params=param, timeout=30, allow_redirects=True)
    if resp.status_code != 200:
        return None
    raw = resp.json()
    inner = raw.get("data", {})
    return inner.get("company", inner) if isinstance(inner, dict) else inner


def get_people(company_id, page_size=20):
    resp = requests.get(f"{BASE_URL}/people/", headers=HEADERS, params={
        "company_ids": company_id, "roles": "founder,ceo,key_person",
        "contact_types": "email", "page_size": page_size
    }, timeout=30, allow_redirects=True)
    if resp.status_code != 200:
        return []
    return resp.json().get("data", {}).get("people", [])


def resolve_investor_names(investor_ids):
    names = []
    for inv_id in investor_ids:
        resp = requests.get(f"{BASE_URL}/investor/", headers=HEADERS, params={"id": inv_id}, timeout=30, allow_redirects=True)
        if resp.status_code == 200:
            inv = resp.json().get("data", {}).get("investor", {})
            if inv.get("name"):
                names.append(inv["name"])
    return names


def extract_company_data(company, domain=""):
    website = f"https://{domain}" if domain else ""
    industries_raw = company.get("industries", [])
    loc = company.get("location", {}) or {}
    deal = company.get("latest_deal", {}) or {}
    deal_date_raw = deal.get("date", "")
    deal_date = deal_date_raw[:10] if deal_date_raw else ""
    days_since = ""
    if deal_date:
        try:
            days_since = (datetime.now() - datetime.strptime(deal_date, "%Y-%m-%d")).days
        except:
            pass
    deal_amount = ""
    for fin in deal.get("financings", []):
        if fin.get("size_usd"):
            deal_amount = fin["size_usd"]
            break
    return {
        "company_name": company.get("name", ""), "website": website,
        "company_linkedin": company.get("linkedin", ""),
        "employees": company.get("num_employees", ""),
        "description": company.get("short_description", ""),
        "industry": industries_raw[0].get("name", "") if industries_raw else "",
        "sub_industry": industries_raw[1].get("name", "") if len(industries_raw) > 1 else "",
        "hq_city": (loc.get("city") or {}).get("name", ""),
        "hq_state": (loc.get("state") or {}).get("name", ""),
        "hq_country": (loc.get("country") or {}).get("name", ""),
        "total_raised": company.get("total_raised"),
        "num_rounds": company.get("num_funding_rounds"),
        "deal_type": deal.get("type", ""), "deal_date": deal_date,
        "days_since": days_since, "deal_amount": deal_amount,
        "deal_investor_ids": deal.get("investors", []),
    }


def build_company_row(cd, investors_str):
    return {
        "Company": cd["company_name"], "Website": cd["website"],
        "Company LinkedIn": cd["company_linkedin"], "Industry": cd["industry"],
        "Sub Industry": cd["sub_industry"], "Employee Count": cd["employees"],
        "Company HQ City": cd["hq_city"], "Company HQ State": cd["hq_state"],
        "Company HQ Country": cd["hq_country"], "Total Raised": cd["total_raised"],
        "Funding Rounds": cd["num_rounds"], "Last Funding Type": cd["deal_type"],
        "Last Funding Date": cd["deal_date"], "Days Since Funding": cd["days_since"],
        "Last Funding Amount": cd["deal_amount"], "Last Deal Investors": investors_str,
        "Description": cd["description"],
    }


def build_lead_row(p, cd, investors_str):
    contact_city, contact_state, contact_country = normalize_location(p.get("city", ""))
    return {
        "Name": p.get("name", ""), "Email": p.get("email", ""),
        "Role": p.get("title", ""), "Company": cd["company_name"],
        "LinkedIn": p.get("linkedin_url", ""), "Website": cd["website"],
        "Company LinkedIn": cd["company_linkedin"], "Phone": p.get("phone", ""),
        "Industry": cd["industry"], "Sub Industry": cd["sub_industry"],
        "Contact City": contact_city, "Contact State": contact_state,
        "Contact Country": contact_country,
        "Company HQ City": cd["hq_city"], "Company HQ State": cd["hq_state"],
        "Company HQ Country": cd["hq_country"], "Employee Count": cd["employees"],
        "Description": cd["description"], "Total Raised": cd["total_raised"],
        "Funding Rounds": cd["num_rounds"], "Last Funding Type": cd["deal_type"],
        "Last Funding Date": cd["deal_date"], "Days Since Funding": cd["days_since"],
        "Last Funding Amount": cd["deal_amount"], "Last Deal Investors": investors_str,
    }


def build_tables(company_urls):
    company_rows, lead_rows = [], []
    for url in company_urls:
        label = url.replace("https://", "").replace("http://", "").replace("www.", "").rstrip("/")
        print(f"Processing: {label}...", end=" ")
        company = resolve_company(url)
        if not company:
            print("NOT FOUND")
            continue
        company_id = company.get("id", "")
        domain = company.get("domain", "")
        cd = extract_company_data(company, domain)
        inv_ids = cd["deal_investor_ids"]
        if inv_ids and isinstance(inv_ids[0], str) and len(inv_ids[0]) > 20:
            print(f"resolving {len(inv_ids)} investors...", end=" ")
            investors_str = ", ".join(resolve_investor_names(inv_ids))
        else:
            investors_str = ""
        company_rows.append(build_company_row(cd, investors_str))
        if not company_id:
            print("no company ID")
            continue
        people = get_people(company_id)
        print(f"{cd['company_name']} -> {len(people)} contacts")
        for p in people:
            lead_rows.append(build_lead_row(p, cd, investors_str))
    return pd.DataFrame(company_rows), pd.DataFrame(lead_rows)

print(f"ScrapingDog API: {'configured' if SCRAPINGDOG_API_KEY else 'NOT SET (size verification disabled)'}")

---
## Option 1: Search by Filters

All filters are optional. Leave blank to skip.  
Use your **taxonomy names** directly — they auto-convert to Fundable slugs.

**Employee counts:** `2-10`, `11-50`, `51-200`, `201-500`, `501-1,000`, `1,001-5,000`, `5,001-10,000`, `10,001+`

**Financing types:** `seed`, `series-a`, `series-b`, `series-c`, `ipo`, `debt`, `grant`

In [ ]:
# === FILTERS (leave blank to skip) ===
NUM_LEADS = 20                              # number of companies (1+ lead per company, total leads >= this)
FUNDING_DAYS = ""                           # funded in last X days (leave blank to skip)

# Use your taxonomy names directly (comma-separated):
INDUSTRIES = "Cryptocurrency"
EMPLOYEE_COUNT = "2-10"

# Funding type (comma-separate, uppercase):
FINANCING_TYPES = ""

# Person location filter — leave ALL blank to skip location filtering:
PERSON_CITY = ""                            # e.g. "San Francisco,New York"
PERSON_STATE = ""                           # e.g. "California,New York"
PERSON_COUNTRY = "United States,Indonesia,Australia,China"


In [ ]:
# Step 1: Convert taxonomy names to Fundable slugs
industry_slugs = to_industry_slugs(INDUSTRIES)
employee_slugs = to_employee_slugs(EMPLOYEE_COUNT)

if industry_slugs:
    print(f"Industries: {INDUSTRIES} -> {industry_slugs}")
if employee_slugs:
    print(f"Employee count: {EMPLOYEE_COUNT} -> {employee_slugs}")

# Step 2: Build API params
params = {
    "roles": "founder,ceo,key_person",
    "contact_types": "email",
    "page_size": 100,
}
if FUNDING_DAYS:
    params["deal_start_date"] = (datetime.now() - timedelta(days=FUNDING_DAYS)).strftime("%Y-%m-%d")
if employee_slugs:
    params["employee_count"] = employee_slugs
if industry_slugs:
    params["industries"] = industry_slugs
if FINANCING_TYPES:
    params["financing_types"] = FINANCING_TYPES

# Parse person location filters
filter_cities = [c.strip().lower() for c in PERSON_CITY.split(",") if c.strip()] if PERSON_CITY else []
filter_states = [s.strip().lower() for s in PERSON_STATE.split(",") if s.strip()] if PERSON_STATE else []
filter_countries = [c.strip().lower() for c in PERSON_COUNTRY.split(",") if c.strip()] if PERSON_COUNTRY else []
has_location_filter = bool(filter_cities or filter_states or filter_countries)

print(f"\nSearching with: {params}")
if has_location_filter:
    print(f"Person location filter: city={filter_cities or 'any'}, state={filter_states or 'any'}, country={filter_countries or 'any'}")
print(f"Target: {NUM_LEADS} companies with 1+ lead each\n")

In [ ]:
# Step 3: Paginate, verify size via LinkedIn, filter location, until target met
company_rows = []
lead_rows = []
companies_kept = 0
companies_checked = 0
companies_size_failed = 0
total_leads = 0
seen_company_ids = set()
page = 0
api_total = None
verify_size = bool(EMPLOYEE_COUNT and SCRAPINGDOG_API_KEY)

if verify_size:
    print(f"Employee size verification: ENABLED (via ScrapingDog + LinkedIn)")
else:
    if EMPLOYEE_COUNT and not SCRAPINGDOG_API_KEY:
        print(f"Employee size verification: DISABLED (SCRAPINGDOG_API_KEY not set, trusting Fundable)")
    print()

while companies_kept < NUM_LEADS:
    params["page"] = page
    resp = requests.get(f"{BASE_URL}/people/", headers=HEADERS, params=params, timeout=30, allow_redirects=True)
    if resp.status_code != 200:
        print(f"API error: {resp.status_code} - {resp.text[:300]}")
        break
    data = resp.json()
    people_batch = data.get("data", {}).get("people", [])
    api_total = data.get("meta", {}).get("total_count", 0)

    if not people_batch:
        print(f"No more results from API (total available: {api_total})")
        break

    # Extract unique companies from this page
    page_companies = {}
    for p in people_batch:
        c = p.get("company", {})
        cid = c.get("id", "")
        if cid and cid not in seen_company_ids:
            if cid not in page_companies:
                page_companies[cid] = c

    for cid, basic in page_companies.items():
        if companies_kept >= NUM_LEADS:
            break

        seen_company_ids.add(cid)
        domain = basic.get("domain", "")
        companies_checked += 1
        company_name = basic.get("name", "?")
        print(f"[{companies_checked}] {company_name} ({domain})...", end=" ")

        # Get leads first, check location before expensive size verification
        people = get_people(cid)
        if not people:
            print("no leads, skipping")
            continue

        # Filter by location first (free, no API calls)
        if has_location_filter:
            matched_people = []
            for p in people:
                p_city, p_state, p_country = normalize_location(p.get("city", ""))
                city_ok = not filter_cities or any(fc in p_city.lower() for fc in filter_cities)
                state_ok = not filter_states or any(fs in p_state.lower() for fs in filter_states)
                country_ok = not filter_countries or any(fc in p_country.lower() for fc in filter_countries)
                if city_ok and state_ok and country_ok:
                    matched_people.append(p)
            if not matched_people:
                print(f"{len(people)} leads, 0 match location, skipping")
                continue
            print(f"{len(people)} leads, {len(matched_people)} match location...", end=" ")
            people = matched_people
        else:
            print(f"{len(people)} leads...", end=" ")

        # Get full company details
        company = resolve_company(domain) if domain else None
        if not company:
            print("company lookup failed")
            continue

        cd = extract_company_data(company, domain)

        # Fill HQ from LinkedIn if Fundable returned empty
        if not cd["hq_city"] and SCRAPINGDOG_API_KEY:
            li_city, li_state, li_country = extract_hq_from_linkedin(cd["company_linkedin"], cd["company_name"])
            if li_city:
                cd["hq_city"] = li_city
                cd["hq_state"] = li_state
                cd["hq_country"] = li_country
                print(f"HQ filled ({li_city}, {li_state})...", end=" ")

        # Verify employee size via LinkedIn (only after location check passes)
        if verify_size:
            linkedin_size, size_match, source = verify_employee_size_via_linkedin(
                cd["company_linkedin"], cd["company_name"], EMPLOYEE_COUNT, cd["employees"]
            )
            if source == "linkedin":
                if size_match:
                    print(f"size OK ({linkedin_size})...", end=" ")
                    cd["employees"] = linkedin_size
                else:
                    print(f"size MISMATCH (LinkedIn: {linkedin_size}, wanted: {EMPLOYEE_COUNT}), skipping")
                    companies_size_failed += 1
                    continue
            else:
                print(f"size unverified ({source}), using Fundable ({cd['employees']})...", end=" ")

        # Resolve investors
        inv_ids = cd["deal_investor_ids"]
        if inv_ids and isinstance(inv_ids[0], str) and len(inv_ids[0]) > 20:
            investors_str = ", ".join(resolve_investor_names(inv_ids))
        else:
            investors_str = ""

        company_rows.append(build_company_row(cd, investors_str))
        for p in people:
            lead_rows.append(build_lead_row(p, cd, investors_str))

        companies_kept += 1
        total_leads += len(people)
        print(f"-> KEPT ({companies_kept}/{NUM_LEADS} companies, {total_leads} total leads)")

    page += 1
    if page * 100 >= (api_total or 0):
        print(f"\nReached end of API results ({api_total} total people)")
        break

company_df = pd.DataFrame(company_rows)
leads_df = pd.DataFrame(lead_rows)
print(f"\n{'='*60}")
if companies_kept < NUM_LEADS:
    print(f"Warning: Only found {companies_kept}/{NUM_LEADS} companies (checked {companies_checked})")
if companies_size_failed:
    print(f"Size verification: {companies_size_failed} companies rejected (wrong size on LinkedIn)")
print(f"Result: {companies_kept} companies | {total_leads} total leads")

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", None)
company_df

In [ ]:
leads_df

---
## Intent Enrichment

Run intent research on discovered companies. Uses Perplexity (sonar-pro) + ScrapingDog (jobs/news).

**Requires:** `OPENROUTER_API_KEY` and `SCRAPINGDOG_API_KEY` in `.env`

In [ ]:
# === INTENT INPUTS ===
PRODUCT_DESCRIPTION = "We help crypto asset managers and professional traders scale managed account businesses without adding operational overhead. Our platform reduces manual work across client onboarding, white-label delivery, bulk trade execution, account monitoring, and performance reporting. Buyers are typically founders, CIOs, heads of trading, portfolio managers, and operations leads responsible for managing multiple client portfolios across exchange accounts."
REQUEST_DESCRIPTION = "Boutique to mid-sized crypto asset managers, managed account providers, hedge funds, family offices, and professional traders managing multiple client portfolios on centralized exchanges. They offer systematic or institutional-grade crypto strategies and need secure onboarding, multi-account execution, centralized account management, and reporting at scale."
INTENT_SIGNALS = "Mentions managed accounts, copy trading, or non-custodial portfolio management; promotes institutional-grade, systematic strategies; states that client assets remain on client-owned exchange or brokerage accounts via secure API integration; uses trust language such as your assets on your exchange; launches new managed products or strategies; hires for trading, operations, onboarding, or client success roles; and highlights reporting, transparency, fee workflows, or growth across clients, exchanges, or geographies."


In [ ]:
import sys, os
os.chdir('/Users/tasnimul/Desktop/Intent_model')
sys.path.insert(0, '/Users/tasnimul/Desktop/Intent_model')

# Create target_fit_model package so all internal imports work
import types, importlib
if 'target_fit_model' not in sys.modules:
    pkg = types.ModuleType('target_fit_model')
    pkg.__path__ = ['/Users/tasnimul/Desktop/Intent_model']
    sys.modules['target_fit_model'] = pkg
    # Register all submodules
    for mod_name in ['config', 'openrouter', 'scrapingdog', 'scoring', 'query',
                     'role_discovery', 'icp_extract', 'intent_enrichment',
                     'discovery', 'model', 'funding_check', 'evidence']:
        try:
            mod = importlib.import_module(mod_name)
            sys.modules[f'target_fit_model.{mod_name}'] = mod
        except ImportError:
            pass

from intent_enrichment import research_company_intent, compute_intent_score_from_signals, compute_lead_score, _cross_source_boost
from openrouter import chat_completion
from config import ICP_PARSER_MODEL as GEMINI_MODEL
from evidence import build_evidence
from scoring import compute_fit_score

if not PRODUCT_DESCRIPTION or not INTENT_SIGNALS:
    print('Set PRODUCT_DESCRIPTION and INTENT_SIGNALS above first')
else:
    # Split original signals by comma (same as UI)
    original_signal_list = [s.strip() for s in INTENT_SIGNALS.split(",") if s.strip()]
    
    # Expand intent signals via Gemini (same as UI)
    expanded_intent = INTENT_SIGNALS
    print("Expanding intent signals via LLM...", end=" ")
    try:
        expand_prompt = f"""You are a B2B sales intelligence expert. A company sells this product:

Product: {PRODUCT_DESCRIPTION}

The user wants to find companies showing these buying signals:
{INTENT_SIGNALS}

Expand this list significantly. Think about ALL possible signals that could indicate a company needs this product. Include:
- Direct signals (exactly what was listed)
- Indirect signals (related behaviors that suggest need)
- Growth signals (expansion, funding, new markets, headcount growth)
- Leadership signals (new executives, restructuring, leadership changes)
- Digital signals (new tools adopted, website changes, job postings)
- Content signals (blog posts, social media, webinars about related topics)
- Financial signals (revenue changes, funding rounds, acquisitions)
- Competitive signals (competitors using similar products)
- Pain signals (negative reviews, complaints, challenges mentioned publicly)

IMPORTANT: Keep the user's original signals FIRST and unchanged. Then add 10-15 high-quality expanded signals. Total 20-25 signals. Quality over quantity — only add signals that are realistically detectable via public web search. No explanations, just the comma-separated list."""
        expanded = chat_completion(prompt=expand_prompt, model=GEMINI_MODEL, temperature=0.3, max_tokens=2000)
        if expanded and len(expanded) > len(INTENT_SIGNALS):
            expanded_intent = expanded.strip()
            print(f"done ({len(INTENT_SIGNALS)} chars -> {len(expanded_intent)} chars)")
        else:
            print("kept original")
    except Exception as e:
        print(f"failed ({e}), using original")
    
    # Build ICP object from user filters (same structure as UI)
    icp_filters = {
        'industries': [i.strip() for i in INDUSTRIES.split(',') if i.strip()] if INDUSTRIES else [],
        'sub_industries': [],
        'employee_counts': [s.strip() for s in EMPLOYEE_COUNT.split(',') if s.strip()] if EMPLOYEE_COUNT else [],
        'countries': [c.strip() for c in PERSON_COUNTRY.split(',') if c.strip()] if PERSON_COUNTRY else [],
        'states': [s.strip() for s in PERSON_STATE.split(',') if s.strip()] if PERSON_STATE else [],
        'cities': [c.strip() for c in PERSON_CITY.split(',') if c.strip()] if PERSON_CITY else [],
        'roles': [],
    }
    
    intent_rows = []
    for _, lead_row in leads_df.iterrows():
        company_name = lead_row['Company']
        # Get company-level intent (cache per company)
        if not hasattr(research_company_intent, '_cache'):
            research_company_intent._cache = {}
        if company_name not in research_company_intent._cache:
            print(f"Researching intent: {company_name}...", end=" ")
            result = research_company_intent(
                company={
                    'company_name': company_name,
                    'website': lead_row.get('Website', ''),
                    'company_linkedin': lead_row.get('Company LinkedIn', ''),
                    'industry': lead_row.get('Industry', ''),
                    'description': lead_row.get('Description', ''),
                    '_funding_evidence': f"Last funding: {lead_row.get('Last Funding Type', '')} on {lead_row.get('Last Funding Date', '')}, Total raised: {lead_row.get('Total Raised', '')}",
                },
                product_description=PRODUCT_DESCRIPTION,
                request_description=REQUEST_DESCRIPTION,
                intent_signals=expanded_intent,
            )
            research_company_intent._cache[company_name] = result
            if result:
                print(f"intent={result.get('intent_score', 0):.2f}")
            else:
                print('FAILED')
        
        result = research_company_intent._cache[company_name]
        signals = result.get('signals', []) if result else []
        raw_intent = compute_intent_score_from_signals(signals, original_signal_list)
        boosted_intent, source_count = _cross_source_boost(signals, raw_intent)
        
        # Compute fit score using scoring.py (same as UI)
        # Find matching industry from all company industries
        all_ind = [i.strip() for i in (lead_row.get('All Industries', '') or lead_row.get('Industry', '')).split(',') if i.strip()]
        icp_ind_lower = {i.lower() for i in icp_filters.get('industries', [])}
        matched_ind = next((i for i in all_ind if i.lower() in icp_ind_lower), lead_row.get('Industry', ''))
        lead_dict = {
            'industry': matched_ind,
            'sub_industry': lead_row.get('Sub Industry', ''),
            'role': lead_row.get('Role', ''),
            'country': lead_row.get('Contact Country', ''),
            'state': lead_row.get('Contact State', ''),
            'city': lead_row.get('Contact City', ''),
            'employee_count': lead_row.get('Employee Count', ''),
            'rep_score': 0,
        }
        fit_score, fit_breakdown = compute_fit_score(lead_dict, icp_filters)
        lead_score = compute_lead_score(boosted_intent, fit_score)
        intent_score = boosted_intent
        intent_paragraph = result.get('intent_paragraph', '') if result else ''
        
        # Build evidence string (same format as UI)
        intent_data_for_evidence = {
            'intent_score': boosted_intent,
            'signals': signals,
            '_source_count': source_count,
        }
        verification_data = {'email_status': 'email_ok', 'stage4_passed': True}
        role_scores = {lead_row.get('Role', ''): fit_breakdown.get('role_match', 0.5)}
        evidence = build_evidence(
            lead=lead_dict,
            icp_filters=icp_filters,
            role_scores=role_scores,
            intent_data=intent_data_for_evidence,
            verification_data=verification_data,
            data_gaps=result.get('_data_gaps', []) if result else [],
        )
        
        intent_rows.append({
            'Name': lead_row.get('Name', ''),
            'Email': lead_row.get('Email', ''),
            'Role': lead_row.get('Role', ''),
            'Company': company_name,
            'LinkedIn': lead_row.get('LinkedIn', ''),
            'Website': lead_row.get('Website', ''),
            'Company LinkedIn': lead_row.get('Company LinkedIn', ''),
            'Phone': lead_row.get('Phone', ''),
            'Industry': lead_row.get('Industry', ''),
            'Sub Industry': lead_row.get('Sub Industry', ''),
            'Contact City': lead_row.get('Contact City', ''),
            'Contact State': lead_row.get('Contact State', ''),
            'Contact Country': lead_row.get('Contact Country', ''),
            'Company HQ City': lead_row.get('Company HQ City', ''),
            'Company HQ State': lead_row.get('Company HQ State', ''),
            'Company HQ Country': lead_row.get('Company HQ Country', ''),
            'Employee Count': lead_row.get('Employee Count', ''),
            'Description': lead_row.get('Description', ''),
            'Intent Details': intent_paragraph,
            'Lead Score': lead_score,
            'Evidence': evidence,
        })
    
    intent_df = pd.DataFrame(intent_rows).sort_values('Lead Score', ascending=False)
    # Clear cache for next run
    research_company_intent._cache = {}
    print(f"\nDone! {len(intent_df)} leads scored across {intent_df['Company'].nunique()} companies")

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
intent_df

---
## Option 2: Search by URLs

Provide company domains or LinkedIn URLs directly.

In [ ]:
# === PUT YOUR URLS HERE ===
company_urls = [
    "stripe.com",
    "ramp.com",
    "notion.so",
]

company_df, leads_df = build_tables(company_urls)
print(f"\nCompanies: {len(company_df)} | Leads: {len(leads_df)}")

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", None)
company_df

In [ ]:
leads_df